In [1]:
#互相关运算
import torch
from torch import nn
from d2l import torch as d2l

def corr2d(X,k):
    """计算二位互相关运算"""
    h,w=k.shape
    Y=torch.zeros((X.shape[0]-h+1),(X.shape[1]-w+1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i,j]=(X[i:i+h,j:j+w]*k).sum()
    return Y

In [2]:
X=torch.tensor([[0.0,1.0,2.0],[3.0,4.0,5.0],[6.0,7.0,8.0]])
k=torch.tensor([[0.0,1.0],[2.0,3.0]])
corr2d(X,k)

tensor([[19., 25.],
        [37., 43.]])

In [3]:
#实现二维卷积层
class Conv2D(nn.Module):
    def __init__(self,kernal_size):
        super().__init__()
        self.weight=nn.Parameter(torch.rand(kernal_size))
        self.bias=nn.Parameter(torch.zeros(1))

    def forward(self,X):
        return corr2d(X,self.weight)+self.bias

In [4]:
#简单应用，定义X,K
X=torch.ones((6,8))
X[:,2:6]=0
X

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

In [5]:
#定义K
k=torch.tensor([[1.0,-1.0]])

In [6]:
#输出Y中的1代表从白色到黑色的边缘，-1代表从黑色到白色的边缘
Y=corr2d(X,k)
Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

In [7]:
#怎么由X和Y学习到k
conv2d=nn.Conv2d(1,1,kernel_size=(1,2),bias=False)

X=X.reshape((1,1,6,8))
Y=Y.reshape((1,1,6,7))

for i in range(10):
    Y_hat=conv2d(X)
    l=(Y_hat-Y)**2
    conv2d.zero_grad()
    l.sum().backward()
    conv2d.weight.data[:]-=conv2d.weight.grad*3e-2
    print(f'batch {i+1},loss {l.sum():.3f}')

batch 1,loss 5.558
batch 2,loss 2.278
batch 3,loss 0.934
batch 4,loss 0.383
batch 5,loss 0.157
batch 6,loss 0.065
batch 7,loss 0.027
batch 8,loss 0.011
batch 9,loss 0.005
batch 10,loss 0.002


In [8]:
#所学的卷积核的权重张量
conv2d.weight.data.reshape((1,2))

tensor([[ 0.9911, -0.9932]])